# Phase 5 — LangGraph: Stateful Agents and Multi-Agent Workflows

## 1. StateGraph Fundamentals
A StateGraph defines nodes (functions) and edges (transitions). State flows through the graph.

In [ ]:
import sys

sys.path.insert(0, "../projects/phase5-langgraph")
from graphs import ResearchState, build_research_graph

graph = build_research_graph()
print("Graph nodes:", list(graph.graph.nodes.keys()))
print("Entry: search → draft → review → (conditional) → draft or END")

## 2. Invoking with Thread IDs
Each thread_id creates an isolated conversation. MemorySaver persists state across invocations.

In [ ]:
initial_state = {
    "query": "LangGraph checkpointing",
    "search_results": [],
    "draft": "",
    "revision_count": 0,
    "approved": False,
}
config = {"configurable": {"thread_id": "notebook-demo-1"}}
result = graph.invoke(initial_state, config=config)
print(f"Draft: {result['draft'][:80]}...")
print(f"Approved: {result['approved']}")
print(f"Revisions: {result['revision_count']}")

## 3. Time-Travel: Inspect Checkpoints

In [ ]:
saved_state = graph.get_state(config)
print("Saved state values:")
print(f"  revision_count: {saved_state.values['revision_count']}")
print(f"  approved: {saved_state.values['approved']}")

## 4. Supervisor Multi-Agent Pattern
A supervisor routes tasks to specialist agents. Each specialist routes back to the supervisor when done.

In [ ]:
from multi_agent import build_supervisor_graph

supervisor = build_supervisor_graph()
for task in ["research quantum computing", "analyze the results", "write the final report"]:
    result = supervisor.invoke(
        {"messages": [], "next_agent": "", "task": task, "results": []}
    )
    print(f"Task: {task!r}")
    print(f"  Result: {result['results']}")

## 5. Human-in-the-Loop (HIL)
interrupt_before pauses graph execution so a human can review before proceeding.

In [ ]:
print("HIL pattern — code snippet (requires PostgresSaver for production):")
print('"""')
print("# Build graph with interrupt")
print("app = graph.compile(checkpointer=saver, interrupt_before=['review'])")
print("# First invocation — pauses at 'review' node")
print("app.invoke(state, config={'configurable': {'thread_id': 'hil-1'}})")
print("# Human reviews, then resumes with Command(resume=...)")
print("from langgraph.types import Command")
print("app.invoke(Command(resume='approved'), config)")
print('"""')

## Key Takeaways
- StateGraph gives full control over routing, state, and failure recovery
- Thread IDs isolate conversations; MemorySaver persists within a session
- interrupt_before enables human approval workflows
- The supervisor pattern is the right default for most multi-agent systems
- LangGraph traces show per-node timing in LangSmith